In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from essentials import resample_data, three_class_labels, two_class_labels, normalize
from features_acc_gyr_improved import GenerateFeatures 
import copy
import json
from sklearn.model_selection import GroupShuffleSplit

In [5]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_dict_imu.pkl', 'rb') as f: 
    imu_dict = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_minze_dict.pkl', 'rb') as f:
    ground_truth_dict = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_dict_urineestimate_method1.pkl', 'rb') as f:
    urine_estimate_dict = pickle.load(f)

Remove the data with the almost the entire data being void.
1. subj_9_void4
2. subj_11_void2

In [6]:
del imu_dict['subj_9_void4']
del imu_dict['subj_11_void2']

In [10]:
imu_dict['subj_10_void1']

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,time,Real time
0,-0.791876,-12.477023,-12.914744,-76.899802,641.745164,76.657393,0.000000,1900-01-01 00:00:00.000000
1,-0.906537,-12.537526,-12.924966,-75.394523,636.162531,187.117862,0.016934,1900-01-01 00:00:00.016934
2,-1.008051,-12.626088,-12.817834,-66.886503,631.233913,280.502795,0.033868,1900-01-01 00:00:00.033868
3,-1.058760,-12.686943,-12.607374,-51.266134,622.780270,413.285740,0.050802,1900-01-01 00:00:00.050802
4,-1.109980,-12.762460,-12.448382,-42.626193,604.685850,626.397577,0.067736,1900-01-01 00:00:00.067736
...,...,...,...,...,...,...,...,...
2785,0.851887,-12.934666,-12.803083,24.121754,-14.021276,73.594612,47.161050,1900-01-01 00:00:47.161050
2786,0.832775,-12.934013,-12.844118,24.730915,-13.687408,40.349366,47.177984,1900-01-01 00:00:47.177984
2787,0.840934,-12.928582,-12.858545,17.845086,-13.745321,-11.086126,47.194918,1900-01-01 00:00:47.194918
2788,0.864205,-12.904527,-12.849104,13.279712,-13.276616,-37.835873,47.211852,1900-01-01 00:00:47.211852


In [ ]:
imu_dict_with_ids = {}
dict = copy.deepcopy(imu_dict)
for exp_id, void_instance in tqdm(enumerate(dict.keys()), desc="Adding experiment ids to IMU data"):
    acc = dict[void_instance]
        
    # Add the experiment id
    acc['experiment_id'] = exp_id + 1  
    
    imu_dict_with_ids[void_instance] = acc

Adding experiment ids to IMU data: 41it [00:00, 9729.91it/s]


In [17]:
imu_df_with_ids = pd.DataFrame()
dict = copy.deepcopy(imu_dict)

for exp_id, void_instance in tqdm(enumerate(dict.keys()), desc="Adding experiment ids to IMU data"):
    acc = dict[void_instance]
        
    # Add the experiment id
    acc['experiment_id'] = exp_id + 1
    
    # appeding to the DataFrame
    imu_df_with_ids = pd.concat([imu_df_with_ids, acc], ignore_index=True)

Adding experiment ids to IMU data: 41it [00:00, 390.18it/s]


In [19]:
imu_df_with_ids

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,time,Real time,experiment_id
0,0.066242,-13.923373,-17.075456,-38.322404,8.725663,83.141039,0.000000,1900-01-01 00:00:00.000000,1
1,0.042000,-13.944164,-17.017063,-26.325645,2.767256,81.080344,0.016935,1900-01-01 00:00:00.016935,1
2,0.006946,-13.985796,-16.976153,-12.871816,-2.367739,91.281087,0.033871,1900-01-01 00:00:00.033871,1
3,-0.012306,-14.016341,-16.957208,-4.652553,-4.760758,130.487448,0.050806,1900-01-01 00:00:00.050806,1
4,0.014324,-13.933079,-16.921674,3.602771,-1.295901,187.665522,0.067741,1900-01-01 00:00:00.067741,1
...,...,...,...,...,...,...,...,...,...
144265,-0.415704,-9.772312,-5.398604,-5.791146,-39.759699,5.077117,41.110177,1900-01-01 00:00:41.110177,41
144266,-0.447311,-9.756686,-5.420740,-14.286152,-42.545283,-8.623367,41.126721,1900-01-01 00:00:41.126721,41
144267,-0.432375,-9.771661,-5.388947,-23.709813,-46.282834,-24.237577,41.143264,1900-01-01 00:00:41.143264,41
144268,-0.405955,-9.756496,-5.326923,-19.426838,-51.825114,-38.346962,41.159807,1900-01-01 00:00:41.159807,41


In [ ]:
#'groups' is the df['experiment_id'] column

# --- Step 1: Split your data by experiment ID ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(imu_df_with_ids, groups=imu_df_with_ids['experiment_id']))

train_df = imu_df_with_ids.iloc[train_idx]
test_df = imu_df_with_ids.iloc[test_idx]

# --- Step 2: Calculate and save parameters ONLY from the training set ---
cols_to_normalize = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']

# Calculate means and stds from the combined training data
train_means = train_df[cols_to_normalize].mean()
train_stds = train_df[cols_to_normalize].std()

# Store parameters in a dictionary
norm_params = {
    'means': train_means.to_dict(),
    'stds': train_stds.to_dict()
}

# Save the dictionary to a JSON file
with open('normalization_parameters.json', 'w') as f:
    json.dump(norm_params, f, indent=4)

print("Normalization parameters saved!")
print(norm_params)

# --- Step 3: Apply the saved parameters everywhere ---

def apply_normalization(df_to_norm: pd.DataFrame, params: dict) -> pd.DataFrame:
    means = params['means']
    stds = params['stds']
    
    for col in cols_to_normalize:
        df_to_norm[col] = (df_to_norm[col] - means[col]) / stds[col]
        
    return df_to_norm

# Apply the same transformation to both train and test sets
train_df_normalized = apply_normalization(train_df.copy(), norm_params)
test_df_normalized = apply_normalization(test_df.copy(), norm_params)

# Now you can proceed with feature extraction on these normalized dataframes